# LoRA & QLoRAAdapters

**Module 9 · Lesson 9.5**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- LoRA From Scratch - the Whole Trick in 15 Lines
- Rank, Alpha & the Learning-Rate Trap
- QLoRA - Quantize the Frozen Base to 4-bit
- What Actually Matters: LR > Targets > Rank
- PEFT in Practice - the Current TRL Pipeline
- LoRA Variants - DoRA, rsLoRA, PiSSA, VeRA

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install transformers torch "trl>=1.0" peft transformers datasets peft torch python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The 65,000 Parameters That Beat 7 Billion

> 💡 **Info**
>
> Here is a 4096×4096 layer - **16.8 million** weights. Full fine-tuning would update every one of them, and a whole 7B model of these needs a multi-GPU rental. Instead we **freeze** the layer and bolt on two small matrices: A (4096×8) and B (8×4096). That is **65,536 trainable parameters - 0.39%**. Train only those, and once the learning rate is right you match full fine-tuning on most tasks. The base never changes; the fine-tune is a ~59MB file you can carry in your pocket. This lesson is *why that works* - and what actually moves the needle.

### 🎯 What you will be able to do after this lesson

- **Build** a LoRA layer from scratch (freeze W0, add B·A, scale α/r) and the current QLoRA + PEFT + TRL pipeline (BitsAndBytesConfig NF4 → LoraConfig all-linear + DoRA → SFTTrainer with processing_class).

- **Compare** the three knobs the research ranks - learning rate, target modules, rank - and why α/r vs α/√r (rsLoRA) changes the effective learning rate.

- **Operate** QLoRA internals: NF4 (quantile-optimal 4-bit), double quantization, paged optimizers, and the bf16-vs-fp16 compute-dtype choice.

- **Evaluate** the adapter lifecycle: variants (DoRA/rsLoRA/PiSSA/VeRA), merge vs multi-LoRA serving, and the India multilingual pattern (Airavata; language-specific > joint).

> 📦 **Info**
>
> ✅ Before you startThis is the mechanism-depth companion to **Lesson 9.4** (the open-source tooling). There you ran LoRA/QLoRA; here you learn how they work. You should know what a LoRA adapter is (9.1) and the QLoRA fit trick (9.4). Merge/eval/deploy hands-on beyond the overview is **Lesson 9.6**.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🔖 **Analogy**
>
> **LoRA is sticky-notes on a frozen textbook.** You do not rewrite the book (the pretrained weights) - you add annotations in the margins that adjust how you read it. The notes are tiny compared to the book, you can peel them off or swap in a different set, and the book underneath is untouched. **Where it breaks down:** too few sticky-notes or notes written too faintly (a too-small learning rate, or annotating only the attention chapters) and the reading barely changes - the annotations have to be bold enough and cover the right pages to matter.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Raise the rank for more quality.”** Rank is the LEAST important knob. Once the learning rate is swept, ranks 8–512 land within ~0.4% of each other.
> - **“LoRA's learning rate is like full fine-tuning's.”** It is ~10× HIGHER. This one mistake explains most disappointing LoRA runs.

> 💡 **Info**
>
> ⚠️ Anti-patternTargeting only attention (q_proj, v_proj) AND leaving the learning rate un-retuned after you change the rank. That is the combination that makes LoRA look worse than it is - MLP layers matter more than attention, and α/r silently rescaled your effective LR.

---

## 🎯 Concept 1: LoRA From Scratch - the Whole Trick in 15 Lines

### LoRA From Scratch - the Whole Trick in 15 Lines

Freeze the weights; add B·A; scale by α/r. B starts at zero so the adapter begins as a no-op.

The insight behind LoRA is that the *change* a fine-tune makes to a weight matrix is empirically low-rank - it can be written as the product of two skinny matrices. **Why is the update low-rank?** A pretrained model already sits close to a good solution, so adapting it to a new task only nudges the weights along a handful of directions, not all of them. Aghajanyan et al. (2021) measured this “intrinsic dimension” and found it is surprisingly tiny, and the LoRA paper showed the update ΔW has a very low effective rank even though the original W0 is full-rank. If ΔW really only needs r directions, you can represent it as B·A with inner dimension r and lose almost nothing. So we never touch the big frozen matrix W0. We add a detour: push the input through A (which squeezes it down to rank r), then B (which expands it back), scale it, and add it to the frozen output.

> 🧩 **Analogy**
>
> It is a **bypass road** around a frozen highway. The highway (W0) is fixed. A is the narrow on-ramp that funnels traffic down to r lanes; B is the off-ramp back to full width. Widen the ramp (raise r) and more traffic can take the detour - but the highway itself never gets repaved.

Both LoRA matrices could be random at the start. Why is B initialized to ZERO?

**📝 `01_lora_scratch.py`** - *PyTorch*

In [ ]:
# LoRA FROM SCRATCH - the whole mechanism in ~15 lines (runnable with torch).
import torch
import torch.nn as nn

class LoRALayer(nn.Module):
    """Wrap a frozen linear layer: output = W0(x) + scale * (x @ A @ B)."""
    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        self.base = base
        self.base.weight.requires_grad = False        # FREEZE the pretrained weights
        if self.base.bias is not None:
            self.base.bias.requires_grad = False      # freeze the bias too (nn.Linear defaults to bias=True)
        d_in, d_out = base.in_features, base.out_features
        self.A = nn.Parameter(torch.randn(d_in, rank) * 0.01)   # small random
        self.B = nn.Parameter(torch.zeros(rank, d_out))         # ZERO -> adapter starts as a no-op
        self.scale = alpha / rank                      # this scalar is the whole "alpha/r" story

    def forward(self, x):
        return self.base(x) + (x @ self.A @ self.B) * self.scale

d = 4096
lora = LoRALayer(nn.Linear(d, d), rank=8, alpha=16)
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in lora.parameters() if not p.requires_grad)
print(f"frozen W0: {frozen:,}  |  trainable A+B: {trainable:,}  ({trainable/frozen*100:.2f}%)")

# Output: (runnable with torch installed)
# frozen W0: 16,781,312  |  trainable A+B: 65,536  (0.39%)

- `base.weight.requires_grad = False` freezes the pretrained matrix - no gradients flow into it.
- `A` is small-random and `B` is zeros, so `x @ A @ B` is 0 at step 0 - the model starts identical to the base.
- `self.scale = alpha / rank` is the entire α/r story (Step 2). It scales how strongly the adapter is applied.
- `forward` returns `base(x) + scale * (x @ A @ B)` - frozen path plus the trained detour.

No torch handy? The parameter count is just arithmetic - A is r×d_in and B is d_out×r, so you train r·(d_in+d_out) instead of d_in·d_out:

**📝 `01b_params.py`** - *runnable*

In [ ]:
# LoRA PARAMETER MATH - no torch needed. A is r x d_in, B is d_out x r.
def lora_params(d_in, d_out, r):
    return r * (d_in + d_out)          # A: r*d_in  +  B: d_out*r
def full_params(d_in, d_out):
    return d_in * d_out

d = 4096
for r in (4, 8, 16, 64):
    lp, fp = lora_params(d, d, r), full_params(d, d)
    print(f"  r={r:<3} one {d}x{d} layer: {lp:>8,} trainable vs {fp:,} full  ({lp/fp*100:.2f}%)")

whole = lora_params(d, d, 16) * 7 * 32   # ~7 all-linear modules x ~32 layers
print(f"  r=16 all-linear across a 7B: ~{whole/1e6:.1f}M trainable (~{whole/7e9*100:.2f}% of 7B) -> a ~{whole*2/1e6:.0f}MB adapter (bf16)")

# Output:
#   r=4   one 4096x4096 layer:   32,768 trainable vs 16,777,216 full  (0.20%)
#   r=8   one 4096x4096 layer:   65,536 trainable vs 16,777,216 full  (0.39%)
#   r=16  one 4096x4096 layer:  131,072 trainable vs 16,777,216 full  (0.78%)
#   r=64  one 4096x4096 layer:  524,288 trainable vs 16,777,216 full  (3.12%)
#   r=16 all-linear across a 7B: ~29.4M trainable (~0.42% of 7B) -> a ~59MB adapter (bf16)

#### 💡 What just happened

⚡ What just happened?A 4096×4096 layer (16.8M frozen params) gained two matrices totalling **65,536 trainable params - 0.39%**. Across a whole 7B with all-linear targets that is ~29.4M trainable (~0.42%), which saves as a ~59MB adapter file instead of a 14GB model. The rank r is the dial: higher r = more capacity and more params, but - as the next step shows - it is *not* the dial that most controls quality.

- Drag rank and hidden-dim; watch params in W vs A+B
- See the % trainable and adapter size shrink

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Rank, Alpha & the Learning-Rate Trap

### Rank, Alpha & the Learning-Rate Trap

The adapter is applied with scale = α/r. That one scalar quietly rescales your effective learning rate as rank changes.

Every LoRA forward multiplies the adapter by `scale = α/r`. With the common convention α = 2r, that scale is a constant 2.0 at every rank - convenient. But the moment you hold α fixed and change r (as many tutorials do), the scale - and therefore the *effective* learning rate the adapter feels - moves with rank. That is the hidden reason “I raised the rank and nothing improved.”

> 🎙️ **Analogy**
>
> α/r is a **volume knob** on the adapter. Raise the rank without adjusting α and you have quietly turned the volume DOWN - the adapter is still there, but you can barely hear it. **rsLoRA** (scale α/√r) is a loudness-compensated knob that keeps the perceived volume steady as you add lanes.

You raise rank from 16 to 256 with α fixed at 16. What happens to the adapter's effective scale?

**📝 `02_scaling.py`** - *runnable*

In [ ]:
# THE alpha/r SCALING - and why "raising rank did nothing".
import math
def scale_lora(alpha, r):   return alpha / r
def scale_rslora(alpha, r): return alpha / math.sqrt(r)

print("  alpha = 2*r (Raschka convention):")
print(f"  {'rank':>5} {'alpha':>6} {'LoRA a/r':>10} {'rsLoRA a/sqrt(r)':>18}")
for r in (8, 16, 64, 256):
    a = 2 * r
    print(f"  {r:>5} {a:>6} {scale_lora(a, r):>10.3f} {scale_rslora(a, r):>18.3f}")

# Now hold alpha FIXED and watch the effective update shrink as rank grows:
a = 16
s16, s256 = scale_lora(a, 16), scale_lora(a, 256)
print(f"  With FIXED alpha={a}: scale at r=16 is {s16:.4f}, at r=256 is {s256:.4f} -> {s16/s256:.0f}x smaller.")
print(f"  That silent {s16/s256:.0f}x shrink IS why 'raising rank did nothing' - re-tune LR or use rsLoRA.")

# Output:
#    rank  alpha   LoRA a/r   rsLoRA a/sqrt(r)
#       8     16      2.000              5.657
#      16     32      2.000              8.000
#      64    128      2.000             16.000
#     256    512      2.000             32.000
#   With FIXED alpha=16: scale at r=16 is 1.0000, at r=256 is 0.0625 -> 16x smaller.
#   That silent 16x shrink IS why 'raising rank did nothing' - re-tune LR or use rsLoRA.

- With α=2r, the LoRA scale is a flat 2.0 at every rank - rsLoRA (α/√r) instead grows with rank.
- With α FIXED at 16, the scale falls from 1.0 (r=16) to 0.0625 (r=256) - a 16× weaker update.
- That silent shrink is the effective-LR trap: the fix is to re-tune the learning rate per rank, or switch to rsLoRA.

#### 💡 What just happened

⚡ What just happened?The scale α/r is not a cosmetic hyperparameter - it multiplies the adapter on every forward pass, so it directly rescales how big a step the adapter takes. Change rank and you have changed your effective learning rate. Either sweep LR when you change rank, or set `use_rslora=True` so the scale is α/√r and stays stable at high rank.

- Set rank and alpha; toggle LoRA vs rsLoRA
- Watch the effective-LR bar move

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: QLoRA - Quantize the Frozen Base to 4-bit

### QLoRA - Quantize the Frozen Base to 4-bit

NF4 puts its 16 levels where the weights actually are. Double-quant and paged optimizers shave the rest.

LoRA already shrinks what you TRAIN. QLoRA shrinks what you STORE: it loads the frozen base in 4-bit so a 7B drops from ~14GB to ~4GB, freeing the GPU to train the bf16 adapters on top. The clever part is the number format. **NF4 (NormalFloat4)** places its 16 quantization levels at the quantiles of a normal distribution, so it spends precision near zero - exactly where trained weights cluster - instead of wasting it in the tails like uniform INT4.

> 📏 **Analogy**
>
> A ruler with only 16 tick marks. INT4 spaces them **evenly** across the whole range - but almost every weight sits near zero, so most ticks are wasted out in the tails. NF4 **crowds the ticks near zero**, where the measurements actually happen, and leaves a few for the rare large values. Same 16 ticks, far better readings where it counts.

Most LLM weights cluster near zero. Which 4-bit format measures them most accurately?

**📝 `03_nf4.py`** - *runnable*

In [ ]:
# NF4 vs INT4, and QLoRA memory. NF4 puts its 16 levels at N(0,1) quantiles.
import math
def norm_ppf(p):   # inverse normal CDF (Acklam) - enough for a demo
    a=[-3.969683028665376e+01,2.209460984245205e+02,-2.759285104469687e+02,1.383577518672690e+02,-3.066479806614716e+01,2.506628277459239e+00]
    b=[-5.447609879822406e+01,1.615858368580409e+02,-1.556989798598866e+02,6.680131188771972e+01,-1.328068155288572e+01]
    c=[-7.784894002430293e-03,-3.223964580411365e-01,-2.400758277161838e+00,-2.549732539343734e+00,4.374664141464968e+00,2.938163982698783e+00]
    dd=[7.784695709041462e-03,3.224671290700398e-01,2.445134137142996e+00,3.754408661907416e+00]
    if p < 0.02425:
        q=math.sqrt(-2*math.log(p));  return (((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5])/((((dd[0]*q+dd[1])*q+dd[2])*q+dd[3])*q+1)
    if p <= 0.97575:
        q=p-0.5; r=q*q;               return (((((a[0]*r+a[1])*r+a[2])*r+a[3])*r+a[4])*r+a[5])*q/(((((b[0]*r+b[1])*r+b[2])*r+b[3])*r+b[4])*r+1)
    q=math.sqrt(-2*math.log(1-p));    return -(((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5])/((((dd[0]*q+dd[1])*q+dd[2])*q+dd[3])*q+1)

raw = [norm_ppf((i + 0.5) / 16) for i in range(16)]
mx = max(abs(v) for v in raw)
nf4  = [round(v / mx, 3) for v in raw]                 # 16 quantile levels, normalized
int4 = [round(-1 + 2*i/15, 3) for i in range(16)]      # 16 UNIFORM levels
near = lambda levels, x: min(levels, key=lambda l: abs(l - x))
weights = [-0.9,-0.4,-0.2,-0.1,-0.05,-0.02,0.0,0.03,0.06,0.1,0.15,0.3,0.5,0.8]   # clustered near 0
e_nf4  = sum(abs(w - near(nf4,  w)) for w in weights) / len(weights)
e_int4 = sum(abs(w - near(int4, w)) for w in weights) / len(weights)
print(f"  NF4 levels near 0: {sorted(l for l in nf4 if abs(l) < 0.2)}")
print(f"  mean abs quant error (near-zero weights):  NF4={e_nf4:.4f}   INT4={e_int4:.4f}  -> NF4 {e_int4/e_nf4:.2f}x lower")
P = 7e9
print(f"  7B base: fp16 ~{P*2/1e9:.1f}GB -> 4-bit NF4 ~{P*0.5/1e9:.1f}GB; double-quant saves ~{P*(0.37/8)/1e9:.1f}GB more -> fits a 16GB T4")

# Output:
#   NF4 levels near 0: [-0.127, -0.042, 0.042, 0.127]
#   mean abs quant error (near-zero weights):  NF4=0.0326   INT4=0.0374  -> NF4 1.15x lower
#   7B base: fp16 ~14.0GB -> 4-bit NF4 ~3.5GB; double-quant saves ~0.3GB more -> fits a 16GB T4

- The NF4 levels bunch up near zero (the ±0.042, ±0.127 ticks) - extra resolution where weights live.
- On a near-zero weight sample, NF4's mean quantization error is lower than uniform INT4's (a modest but real edge on this sample).
- Memory: a 7B base falls from ~14GB (fp16) to ~3.5GB (4-bit); double-quant trims ~0.3GB more - now it fits a 16GB T4 with room to train.

#### 💡 What just happened

⚡ What just happened?QLoRA = three stacked tricks: **NF4** compresses the frozen base to 4-bit (quantile-optimal for weights), **double quantization** compresses even the quantization constants (~0.37 bits/param, no accuracy loss), and **paged optimizers** put the optimizer state in NVIDIA unified (pageable) memory that migrates to CPU RAM during a VRAM spike, so a long batch does not OOM-crash. The compute dtype matters too: **bf16** for SFT because its 8-bit exponent gives the same range as fp32, so large activations and gradients do not overflow the way they can in **fp16** (whose 5-bit exponent has a much narrower range); fp16 is kept for RL, where its smaller gradient magnitudes are reported to be more stable. Net effect: a 7B QLoRA fine-tune runs on a free Colab T4.

- Pick a weight value; choose NF4 / INT4 / FP4
- See which bin it lands in and the quant error

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: What Actually Matters: LR > Targets > Rank

### What Actually Matters: LR > Targets > Rank

The 2024-2026 ablations agree: sweep the learning rate first, target all-linear second, and stop obsessing over rank.

The most useful result in the recent literature is a ranking of the knobs. **“LoRA Without Regret” (Thinking Machines, 2025)** swept rank across three orders of magnitude (r=1 to 512) with a per-condition LR sweep and found LoRA matches full fine-tuning when (a) adapters cover attention AND MLP layers and (b) the learning rate is ~10× higher than full-FT's. For reinforcement learning, even **rank=1** sufficed. “Learning Rate Matters” (2025) found all LoRA variants converge within ~0.43% once LR is properly swept.

> 🎤 **Analogy**
>
> Tuning LoRA is like **mixing a track**. The learning rate is the master fader - get it wrong and nothing else you touch is audible. Target modules are which instruments are in the mix (leave out the MLP ‘drums’ and it sounds thin). Rank is a subtle EQ tweak - nice once everything else is right, but it will never rescue a bad master level.

Your LoRA underperforms full fine-tuning. Which single change most reliably closes the gap?

**📝 `04_impact.py`** - *runnable*

In [ ]:
# WHAT TO FIX FIRST - the 2024-2026 hierarchy encoded: LR > targets > rank.
def diagnose(lr_retuned, targets, rank):
    fixes = []
    if not lr_retuned:
        fixes.append(("LR", "re-tune LR: LoRA's optimum is ~10x full-FT; a rank change shifts the effective LR"))
    if targets == "attention":
        fixes.append(("TARGETS", "use all-linear: MLP matters more than attention; attention-only stunts learning"))
    if rank < 8:
        fixes.append(("RANK", "raise rank to >=8 (rank is the LEAST important knob, but <8 can underfit)"))
    if not fixes:
        fixes.append(("OK", "LR retuned + all-linear + sane rank -> on the frontier; improve DATA next"))
    return fixes

for cfg in [(False, "attention", 4), (False, "all-linear", 16), (True, "attention", 64), (True, "all-linear", 16)]:
    fixes = diagnose(*cfg)
    print(f"  lr_retuned={str(cfg[0]):5} targets={cfg[1]:<10} rank={cfg[2]:<3} -> FIX {fixes[0][0]}: {fixes[0][1]}")

# Output:
#   lr_retuned=False targets=attention  rank=4   -> FIX LR: re-tune LR: LoRA's optimum is ~10x full-FT; a rank change shifts the effective LR
#   lr_retuned=False targets=all-linear rank=16  -> FIX LR: re-tune LR: LoRA's optimum is ~10x full-FT; a rank change shifts the effective LR
#   lr_retuned=True  targets=attention  rank=64  -> FIX TARGETS: use all-linear: MLP matters more than attention; attention-only stunts learning
#   lr_retuned=True  targets=all-linear rank=16  -> FIX OK: LR retuned + all-linear + sane rank -> on the frontier; improve DATA next

- The helper always surfaces the HIGHEST-priority fix first: an un-retuned LR outranks bad targets, which outranks a too-small rank.
- Attention-only targets get flagged even at rank 64 - coverage beats rank.
- Only when LR is retuned AND targets are all-linear does it say ‘improve DATA next’ - the real frontier.

#### 💡 What just happened

⚡ What just happened?Internalize the hierarchy: **learning rate > target modules > rank**. Sweep LR first (LoRA’s optimum is ~10× full-FT’s) **instead of** reaching for rank. Then set `target_modules="all-linear"` so MLP layers are covered - they matter more than attention. Only then, if at all, tune rank. Most “LoRA is worse than full FT” reports are really “I used the full-FT learning rate.”

- Toggle LR-retuned, targets, and rank
- Watch the predicted quality delta

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: PEFT in Practice - the Current TRL Pipeline

### PEFT in Practice - the Current TRL Pipeline

BitsAndBytesConfig NF4 → LoraConfig(all-linear, DoRA) → SFTTrainer(processing_class) → save adapter → merge.

Everything above assembles into one short, current pipeline. The only trap is API drift: **TRL v1.0** renamed `tokenizer=` to `processing_class=` and moved `max_seq_length` onto `SFTConfig` as `max_length`. Old tutorials still show the removed names - use the current ones **instead**, since there is no backward-compatible alias.

Your 2024 training script calls SFTTrainer(tokenizer=tok, max_seq_length=512) on TRL v1.0. What happens?

**📝 `05_peft.py PEFT + TRL`** - *v1.0*

In [ ]:
# PEFT IN PRACTICE - the CURRENT (TRL v1.0) QLoRA pipeline. Illustrative (needs a GPU).
# pip install transformers datasets peft trl bitsandbytes accelerate
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import torch

name = "google/gemma-3-4b-it"          # pick the current model: Gemma 3/4, Llama 3.3/4, Qwen3
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map="auto")
tok = AutoTokenizer.from_pretrained(name)
model = prepare_model_for_kbit_training(model)

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, target_modules="all-linear",   # all-linear >> attention-only
    lora_dropout=0.05, use_dora=True, task_type="CAUSAL_LM"))   # DoRA on (2026 default)
model.print_trainable_parameters()

trainer = SFTTrainer(
    model=model, train_dataset=train_ds,
    processing_class=tok,               # was tokenizer=  (removed in TRL v1.0)
    args=SFTConfig(output_dir="./adapter", num_train_epochs=3, learning_rate=2e-4,
                   lr_scheduler_type="cosine", max_length=2048, bf16=True))  # was max_seq_length
# trainer.train()
# model.save_pretrained("./adapter")           # ~tens of MB, NOT the 14GB base
# merged = model.merge_and_unload()            # fold LoRA into W0 for single-task serving

# Output: (illustrative)
# trainable params: 29,933,568 || all params: 4,300,000,000 || trainable%: 0.70

- `BitsAndBytesConfig(load_in_4bit, nf4, double_quant)` is Step 3's QLoRA base in one call.
- `LoraConfig(target_modules="all-linear", use_dora=True)` applies Step 4's lesson (coverage) and Step 6's default (DoRA).
- `processing_class=tok` and `SFTConfig(max_length=...)` are the TRL v1.0 names - the old `tokenizer=`/`max_seq_length` are gone.
- `save_pretrained` writes a ~tens-of-MB adapter; `merge_and_unload` folds it into W0 for single-task serving.

#### 💡 What just happened

⚡ What just happened?The production recipe is small: load 4-bit, attach an all-linear DoRA adapter, train with the current TRL names at LR 2e-4 (10× full-FT) on a cosine schedule, then save a tiny adapter. Change one thing at a time and you can read every line here back to a concept from Steps 1–4.

---

## 🎯 Concept 6: LoRA Variants - DoRA, rsLoRA, PiSSA, VeRA

### LoRA Variants - DoRA, rsLoRA, PiSSA, VeRA

Each is a one-line PEFT flag. Know which to switch on for SFT, for high rank, and for RL.

Beyond vanilla LoRA, a handful of variants each change one thing. **DoRA** writes each weight column as `W = m · V/||V||` - a scalar *magnitude* m times a unit-length *direction* V/||V||. It trains the magnitude m directly and LoRA-adapts only the direction V. That separation mirrors how full fine-tuning actually moves weights (which change magnitude and direction somewhat independently); it buys ~1–4% and merges to zero inference cost. **rsLoRA** is the α/√r fix from Step 2. **PiSSA** initializes A and B from the top-r SVD of the weight (train the “signal,” converge faster) - but it is reported to FAIL in GRPO/RL. **VeRA** shares one frozen random pair across all layers and learns tiny per-layer scales, for ~10× fewer trainable params.

> 🧰 **Analogy**
>
> A drawer of screwdriver bits. **DoRA** is the everyday bit you leave in the driver (default-on). **rsLoRA** is the long-reach bit for high-rank jobs. **PiSSA** is a fast bit that strips when you use it on the wrong screw (RL). **VeRA** is the tiny travel bit for when weight/space is scarce. Same driver - you just click in the bit the job needs.

You are doing GRPO reinforcement-learning fine-tuning. Which variant should you AVOID?

**📝 `06_variant.py`** - *runnable*

In [ ]:
# WHICH VARIANT? - DoRA / rsLoRA / PiSSA / VeRA as PEFT flags.
def pick_variant(goal, rank, param_budget):
    flags = {"use_dora": True}
    notes = ["use_dora=True (magnitude+direction; ~+1-4%, merges to zero inference cost)"]
    if rank >= 32:
        flags["use_rslora"] = True
        notes.append("use_rslora=True (alpha/sqrt(r) - stable gradients at high rank)")
    if goal == "sft-fast-converge":
        notes.append('init_lora_weights="pissa" (SVD init; faster convergence for SFT)')
    if goal == "rl":
        flags = {"use_dora": False}
        notes = ["vanilla LoRA for RL: PiSSA FAILS in GRPO; rank=1 can suffice (LoRA Without Regret)"]
    if param_budget == "extreme-low":
        notes.append("consider VeRA (shared frozen A,B; ~10x fewer trainable params)")
    return flags, notes

for goal, rank, budget in [("sft",16,"normal"),("sft-fast-converge",64,"normal"),("rl",8,"normal"),("sft",16,"extreme-low")]:
    flags, notes = pick_variant(goal, rank, budget)
    print(f"  goal={goal:18s} rank={rank:<3} budget={budget:12s} -> {flags}")
    for n in notes:
        print(f"      - {n}")

# Output:
#   goal=sft                rank=16  budget=normal       -> {'use_dora': True}
#       - use_dora=True (magnitude+direction; ~+1-4%, merges to zero inference cost)
#   goal=sft-fast-converge  rank=64  budget=normal       -> {'use_dora': True, 'use_rslora': True}
#       - use_dora=True (magnitude+direction; ~+1-4%, merges to zero inference cost)
#       - use_rslora=True (alpha/sqrt(r) - stable gradients at high rank)
#       - init_lora_weights="pissa" (SVD init; faster convergence for SFT)
#   goal=rl                 rank=8   budget=normal       -> {'use_dora': False}
#       - vanilla LoRA for RL: PiSSA FAILS in GRPO; rank=1 can suffice (LoRA Without Regret)
#   goal=sft                rank=16  budget=extreme-low  -> {'use_dora': True}
#       - use_dora=True (magnitude+direction; ~+1-4%, merges to zero inference cost)
#       - consider VeRA (shared frozen A,B; ~10x fewer trainable params)

- SFT at r=16 → DoRA on (the 2026 default). Push rank to 32+ → rsLoRA turns on automatically.
- ‘sft-fast-converge’ adds PiSSA's SVD init; an extreme parameter budget suggests VeRA.
- RL flips the recommendation: vanilla LoRA, DoRA off, PiSSA avoided - the RL literature's guidance.

#### 💡 What just happened

⚡ What just happened?The pragmatic default is **QLoRA + DoRA** (`load_in_4bit=True, use_dora=True`) at r=16 with all-linear targets; add `use_rslora=True` at r≥32, and `init_lora_weights="pissa"` when SFT convergence speed matters. For RL, drop back to vanilla LoRA. Every one of these is a single PEFT config flag - no new library.

- Pick SFT / RL / high-rank / low-budget
- Get the recommended flags and why

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Adapter Lifecycle - Merge vs Multi-LoRA Serving

### Adapter Lifecycle - Merge vs Multi-LoRA Serving

One base can serve many adapters. Merge for a single task; keep separate to route many tasks per request.

A trained adapter is a tiny file. You have two ways to ship it. **Merge** (`merge_and_unload()`) folds B·A back into W0, giving one standalone model with zero inference overhead - ideal for a single task. **Keep it separate** and you can serve ONE base model plus many adapters, routed per request: `vLLM --enable-lora --lora-modules`, or S-LoRA / Punica for thousands of adapters. Merge several adapters into one with TIES / DARE (mergekit).

> 🎟️ **Analogy**
>
> Adapters are **lens filters** for one camera body (the base). **Merging** is fusing a filter permanently onto the lens - great if you only ever shoot that one look. **Multi-LoRA serving** is a filter wheel: keep the body, click in whichever filter each shot needs. One expensive body, many cheap filters, swapped per request.

You have 3 task adapters and want one deployment serving all 3 with independent updates. Best option?

**📝 `07_serve.py`** - *runnable*

In [ ]:
# MERGE vs MULTI-LoRA - the serving memory tradeoff.
def lora_params(d_in, d_out, r): return r * (d_in + d_out)
def base_gb(B):           return B * 2.0                          # fp16 base
def adapter_gb(B, r):     return lora_params(4096,4096,r)*7*32*(B/7.0)*2/1e9
def naive(n, B):          return n * base_gb(B)                   # N merged copies
def multilora(n, B, r):   return base_gb(B) + n * adapter_gb(B, r)  # 1 base + N adapters

B, r = 7, 16
print("  7B base, r=16 adapters. Serving N task adapters:")
print(f"  {'N':>3} {'naive (N copies)':>18} {'multi-LoRA':>14} {'saved':>10}")
for n in (1, 3, 5, 10):
    nv, ml = naive(n, B), multilora(n, B, r)
    print(f"  {n:>3} {nv:>16.1f}GB {ml:>12.1f}GB {nv-ml:>8.1f}GB")
print(f"  one r=16 adapter ~{adapter_gb(B,r)*1000:.0f}MB vs a {base_gb(B):.0f}GB base.")
print(f"  Merge (merge_and_unload) only for a SINGLE task: zero overhead, but you lose per-task routing.")

# Output:
#   7B base, r=16 adapters. Serving N task adapters:
#     N   naive (N copies)     multi-LoRA      saved
#     1             14.0GB         14.1GB     -0.1GB
#     3             42.0GB         14.2GB     27.8GB
#     5             70.0GB         14.3GB     55.7GB
#    10            140.0GB         14.6GB    125.4GB
#   one r=16 adapter ~59MB vs a 14GB base.
#   Merge (merge_and_unload) only for a SINGLE task: zero overhead, but you lose per-task routing.

- At N=1 the two paths tie (just merge). By N=5, naive is 70GB vs multi-LoRA 14.3GB - a 55.7GB saving.
- One r=16 adapter is ~59MB against a 14GB base, so extra tasks cost almost nothing.
- Merge only for a SINGLE task: you get zero overhead but forfeit per-task routing and independent updates.

#### 💡 What just happened

⚡ What just happened?The merge-vs-serve decision follows the task count. **Merge** for one task (zero overhead). **Keep separate** for many: one base + N tiny adapters, routed per request, scales to thousands of adapters (S-LoRA / Punica) with ~ms/token overhead. Combine adapters offline with TIES/DARE only when the tasks are similar and you want a single model.

- Drag adapter count and base size
- Compare naive N-copies vs one base + N adapters

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: India LoRA Patterns - Airavata & Multilingual Adapters

### India LoRA Patterns - Airavata & Multilingual Adapters

LoRA preserves English while adding an Indic language. For many languages, separate adapters beat one joint adapter.

The headline result from Indic fine-tuning is a LoRA argument. **Airavata (AI4Bharat)** fine-tuned OpenHathi-7B with LoRA (r=16, α=32, all projections, LR 5e-4, 4 epochs) and found that LoRA ADDS Hindi while *preserving* English - whereas full fine-tuning on Hindi data DEGRADES English. LoRA's constrained update acts as implicit regularization. For several languages, language-specific adapters beat one joint adapter, because in a joint run the high-resource languages dominate the gradient.

> 🇮🇳 **Analogy**
>
> Building a multilingual model with full FT is **repainting one wall** a new colour - you cover the old one. LoRA is **hanging a new painting** on the wall: the room gains the new work and the original wall is intact. For five languages, five paintings you can rehang independently beat one muddy canvas where the loudest colour wins.

You need adapters for 5 Indian languages, some low-resource. One joint adapter or five separate ones?

**📝 `08_india.py`** - *runnable*

In [ ]:
# INDIA LoRA PLANNER - one joint adapter or many? Airavata's lesson: LoRA preserves English.
def india_plan(n_langs, resource):
    if n_langs == 1:
        return ("single LoRA (Airavata recipe: r=16, alpha=32, all proj, LR 5e-4, 4 epochs)",
                "adds the language while PRESERVING English (full FT on Hindi DEGRADES English)")
    if resource == "low":
        return (f"{n_langs} language-specific LoRA adapters + vLLM multi-LoRA (route by detected language)",
                "language-specific beats a joint adapter: high-resource langs dominate a joint gradient")
    return (f"{n_langs} adapters; evaluate joint vs separate, serve via multi-LoRA",
            "start separate; merge only if the languages are close and you want one model")

for n, res in [(1, "low"), (5, "low"), (2, "high")]:
    stack, why = india_plan(n, res)
    print(f"  langs={n} resource={res:5s} -> {stack}")
    print(f"      why: {why}")
print("  Memory: 1 base (~14GB) + 5 adapters (~295MB) << 5 merged models (~70GB).")

# Output:
#   langs=1 resource=low   -> single LoRA (Airavata recipe: r=16, alpha=32, all proj, LR 5e-4, 4 epochs)
#       why: adds the language while PRESERVING English (full FT on Hindi DEGRADES English)
#   langs=5 resource=low   -> 5 language-specific LoRA adapters + vLLM multi-LoRA (route by detected language)
#       why: language-specific beats a joint adapter: high-resource langs dominate a joint gradient
#   langs=2 resource=high  -> 2 adapters; evaluate joint vs separate, serve via multi-LoRA
#       why: start separate; merge only if the languages are close and you want one model
#   Memory: 1 base (~14GB) + 5 adapters (~295MB) << 5 merged models (~70GB).

- One language → a single Airavata-style adapter that preserves English.
- Several low-resource languages → separate adapters + multi-LoRA routing (no joint-gradient domination).
- Memory: one base + five ~59MB adapters (the Step-1/Step-7 figure) is a fraction of five merged models.

#### 💡 What just happened

⚡ What just happened?For Indic (and any multilingual) work, LoRA is often **strictly preferable** to full fine-tuning: it adds the new language while preserving the base's existing capabilities. The Airavata config (r=16, α=32, all projections) is the proven starting point, and for many languages you train separate adapters and compose them with multi-LoRA serving - the same lifecycle from Step 7.

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> LoRA freezes the base and trains a low-rank detour B·A scaled by α/r (Step 1) - and that scale silently rescales your effective learning rate, which is why rank alone rarely helps (Step 2). QLoRA quantizes the frozen base to 4-bit NF4 so the whole thing fits a small GPU (Step 3). What actually moves quality is **learning rate > target modules > rank** (Step 4). The current TRL pipeline wires it together with the renamed API (Step 5); DoRA/rsLoRA/PiSSA/VeRA are one-flag refinements (Step 6); and you ship it by merging for one task or serving many adapters on one base (Step 7) - the exact pattern that lets one Indian base model speak many languages (Step 8).

> 📦 **Info**
>
> ➡️ Where this goes next**Lesson 9.6 builds on this** - Evaluation, Merging & Deployment takes the adapter you just understood and puts it through eval harnesses, adapter merging (TIES/DARE in depth), and production deployment. The serving math here (merge vs multi-LoRA) becomes hands-on there, and the cost/latency side comes up again in Module 13 and the LLMOps work in Module 14. The reference implementations live in the HuggingFace [PEFT](https://github.com/huggingface/peft) and [TRL](https://github.com/huggingface/trl) repositories.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **LoRA & QLoRAAdapters**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-9_5.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-9.5.html` - regenerate this notebook whenever the source HTML is updated.*
